# Day-1 Player Retention Rate

**Difficulty:** Medium

**Companies:** Meta, Amazon

---

### Problem Description

You are given a table named `pra_player_activity`, which records the daily login activity of players in a game. Each row represents an instance of a player logging in, playing a certain number of games, and logging out.

Write a solution (using PySpark / Spark SQL) to determine the **Day-1 player retention rate**, defined as the proportion of players who logged in again on the calendar day immediately following their first-ever login date.

---

### Schema

#### `pra_player_activity`

| Column Name | Type | Description |
| --- | --- | --- |
| `player_id` | `INT` | Unique identifier for each player |
| `device_id` | `INT` | Identifier for the device used |
| `event_date` | `DATE` | Date of the login activity (`YYYY-MM-DD`) |
| `games_played` | `INT` | Number of games played in that session |

---

### Requirements & Constraints

1. **Denominator:** The total count of distinct players present in the dataset.
2. **Numerator:** The count of distinct players who recorded at least one activity on the day immediately following their earliest recorded `event_date` (`first_login_date + 1 day`).
3. **Deduplication:** Players may have multiple log entries on the same date. Each player must count at most once in the numerator and once in the denominator.
4. **Rounding:** Round the result to **2 decimal places**.
5. **Output Schema:** A single column named `fraction`.

---

### Example

#### Input: `pra_player_activity`

| player_id | device_id | event_date | games_played |
| --- | --- | --- | --- |
| 1 | 2 | 2024-03-01 | 5 |
| 1 | 2 | 2024-03-02 | 6 |
| 2 | 3 | 2024-06-25 | 1 |
| 3 | 1 | 2024-03-02 | 0 |

#### Output

| fraction |
| --- |
| 0.33 |

#### Explanation

* **Player 1:** First login is `2024-03-01`. Logged in next day (`2024-03-02`). $\rightarrow$ **Retained**
* **Player 2:** First login is `2024-06-25`. No login on `2024-06-26`. $\rightarrow$ **Not Retained**
* **Player 3:** First login is `2024-03-02`. No login on `2024-03-03`. $\rightarrow$ **Not Retained**

$$\text{Fraction} = \frac{1 \text{ (Retained)}}{3 \text{ (Total)}} \approx 0.33$$

In [1]:
!pip install -q pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# In Google Colab with PySpark, `df.show()` is typically used to display DataFrames,
# rather than `display(df)`.

In [10]:
import pandas as pd

In [5]:
spark = SparkSession.builder.appName('run_pyspark').getOrCreate()

In [82]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 2. Initialize Spark Session
spark = SparkSession.builder \
    .appName("PlayerRetention") \
    .getOrCreate()

# 3. Define schema and sample data (including edge cases: multiple logins per day)
schema = StructType([
    StructField("player_id", IntegerType(), True),
    StructField("device_id", IntegerType(), True),
    StructField("event_date", StringType(), True),
    StructField("games_played", IntegerType(), True)
])

data = [
    (1, 2, "2024-03-01", 5),
    (1, 2, "2024-03-01", 2),  # Same day duplicate activity
    (1, 2, "2024-03-02", 6),  # Retained on Day 1
    (2, 3, "2024-06-25", 1),  # Not retained
    (3, 1, "2024-03-02", 0)   # Not retained
]

# Create DataFrame and cast event_date to DateType
df = spark.createDataFrame(data, schema=schema) \
    .withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))

# # 4. PySpark Solution (DataFrame API)
# # Step A: Find each player's first login date
# first_logins = df.groupBy("player_id").agg(
#     F.min("event_date").alias("first_date")
# )

# # Step B: Deduplicate overall activity by player and date
# distinct_activity = df.select("player_id", "event_date").distinct()

# # Step C: Left join to check who returned on first_date + 1 day
# joined = first_logins.join(
#     distinct_activity,
#     (first_logins.player_id == distinct_activity.player_id) &
#     (distinct_activity.event_date == F.date_add(first_logins.first_date, 1)),
#     how="left"
# )

# # Step D: Compute the fraction (numerator / denominator)
# result = joined.agg(
#     F.round(
#         F.count(distinct_activity.player_id) / F.count(first_logins.player_id),
#         2
#     ).alias("fraction")
# )

# # 5. Display Result
# result.show()

### Displaying PySpark DataFrame as Pandas DataFrame for Rich Output

To leverage Colab's `display()` function for a more interactive and visually appealing output (similar to what it provides for Pandas DataFrames), you can convert the PySpark DataFrame to a Pandas DataFrame using the `.toPandas()` method.

**Caution:** The `.toPandas()` method collects all the data from the distributed PySpark DataFrame into a single Pandas DataFrame on the driver node. This can consume a significant amount of memory and might lead to out-of-memory errors if your PySpark DataFrame is very large. Use it with care, especially on large datasets, or consider sampling your data before conversion if you only need a preview.

In [83]:
df.show()

+---------+---------+----------+------------+
|player_id|device_id|event_date|games_played|
+---------+---------+----------+------------+
|        1|        2|2024-03-01|           5|
|        1|        2|2024-03-01|           2|
|        1|        2|2024-03-02|           6|
|        2|        3|2024-06-25|           1|
|        3|        1|2024-03-02|           0|
+---------+---------+----------+------------+



In [84]:
def player_retention(input_df):
  retention_window = Window.partitionBy("player_id").orderBy("event_date")

  lead_df = input_df.withColumn("next_event_date", F.lead("event_date",1).over(retention_window))
  lead_df = lead_df.withColumn("next_day_login",
    F.when(F.datediff(F.col("next_event_date"), F.col("event_date")) == 1, F.col("next_event_date"))
    .otherwise(F.lit(None)) # Set to None if not exactly 1 day later
)

  count_df = lead_df.groupBy("player_id").agg(
      F.countDistinct("player_id").alias("first_login"),
      F.countDistinct("next_day_login").alias("next_day_login")
  ).replace(0,None, subset = 'next_day_login')

  # Dynamically count non-nulls for all columns
  non_null_counts = count_df.select([F.count(F.col(c)).alias(c) for c in ['first_login', 'next_day_login']])

  result_df = non_null_counts.withColumn("fraction", F.round(F.col("next_day_login") / F.col("first_login"), 2))
  result_df = result_df.select("fraction")

  return result_df

In [85]:
result = player_retention(df)
result.show()

+--------+
|fraction|
+--------+
|    0.33|
+--------+



## AI Solution

In [86]:
# ----------------------------------------------------
# Method 1: PySpark DataFrame API
# ----------------------------------------------------

# Step A: Find the first login date for each player
first_login_df = df.groupBy("player_id").agg(
    F.min("event_date").alias("first_date")
)

# Step B: Left join back to check if player logged in on first_date + 1 day
# Select specific columns and alias them to avoid ambiguity
retention_df = first_login_df.alias("fl").join(
    df.alias("activity"),
    (F.col("fl.player_id") == F.col("activity.player_id")) &
    (F.col("activity.event_date") == F.date_add(F.col("fl.first_date"), 1)),
    how="left"
).select(
    F.col("fl.player_id").alias("original_player_id"),
    F.col("fl.first_date"),
    F.col("activity.player_id").alias("retained_player_id"), # This will be null if no retention
    F.col("activity.event_date").alias("retained_event_date") # This will be null if no retention
)

# Step C: Calculate fraction (retained distinct players / total distinct players)
# Denominator: Total count of distinct players from original_player_id
total_distinct_players = retention_df.select(F.count_distinct("original_player_id")).collect()[0][0]

# Numerator: Count of distinct players who had a next-day login (retained_player_id is not null)
retained_distinct_players = retention_df.select(F.count_distinct("retained_player_id")).collect()[0][0]

# Calculate the fraction, handling division by zero
fraction_value = 0.0
if total_distinct_players > 0:
    fraction_value = round(retained_distinct_players / total_distinct_players, 2)

# Create a DataFrame for the result as requested by the problem description
result_df = spark.createDataFrame([(fraction_value,)], ["fraction"])

result_df.show()

+--------+
|fraction|
+--------+
|    0.33|
+--------+



## Testing

In [87]:
# Define schema
schema = StructType([
    StructField("player_id", IntegerType(), True),
    StructField("device_id", IntegerType(), True),
    StructField("event_date", StringType(), True),
    StructField("games_played", IntegerType(), True)
])

# -------------------------------------------------------------------------
# Test Cases Data
# -------------------------------------------------------------------------
test_cases = [
    {
        "name": "Test Case 1: Standard / Multi-Player Mix (Expected: 0.33)",
        "expected": 0.33,
        "data": [
            (1, 2, "2024-03-01", 5),
            (1, 2, "2024-03-02", 6),
            (2, 3, "2024-06-25", 1),
            (3, 1, "2024-03-02", 0)
        ]
    },
    {
        "name": "Test Case 2: 100% Retention (All players return next day) (Expected: 1.0)",
        "expected": 1.0,
        "data": [
            (1, 1, "2023-01-10", 3),
            (1, 1, "2023-01-11", 4),
            (2, 2, "2023-05-20", 1),
            (2, 2, "2023-05-21", 2)
        ]
    },
    {
        "name": "Test Case 3: 0% Retention (No player returns on Day 2) (Expected: 0.0)",
        "expected": 0.0,
        "data": [
            (1, 1, "2023-01-10", 2),
            (1, 1, "2023-01-15", 3),  # Logged in, but 5 days later
            (2, 2, "2023-02-01", 0)
        ]
    },
    {
        "name": "Test Case 4: Leap Year & Month Rollover (Expected: 0.5)",
        "expected": 0.5,
        # Player 10: 2024 is a leap year (Feb 29 -> Mar 01) -> Retained
        # Player 20: 2023 is non-leap year (Feb 28 -> Mar 02) -> Skipped Mar 01 -> Not retained
        "data": [
            (10, 1, "2024-02-29", 1),
            (10, 1, "2024-03-01", 4),
            (20, 2, "2023-02-28", 2),
            (20, 2, "2023-03-02", 3)
        ]
    },
    {
        "name": "Test Case 5: New Year Rollover (Expected: 1.0)",
        "expected": 1.0,
        # Dec 31 -> Jan 01 boundary
        "data": [
            (100, 1, "2023-12-31", 10),
            (100, 1, "2024-01-01", 5)
        ]
    },
    {
        "name": "Test Case 6: Duplicate Logins on Both Days & Later Consecutive Days (Expected: 0.5)",
        "expected": 0.5,
        # Player 1: Multiple logins on day 1 & day 2 -> should only count ONCE -> Retained
        # Player 2: First login Day 1, then consecutive on Day 4 & Day 5 -> NOT retained (missed Day 2)
        "data": [
            (1, 1, "2024-01-01", 1),
            (1, 2, "2024-01-01", 3),
            (1, 1, "2024-01-02", 2),
            (1, 3, "2024-01-02", 0),
            (2, 1, "2024-01-01", 5),
            (2, 1, "2024-01-04", 2),
            (2, 1, "2024-01-05", 3)
        ]
    }
]

# -------------------------------------------------------------------------
# Your Solution Function (Insert your PySpark code here)
# -------------------------------------------------------------------------
# def solve_retention(df):
#     """
#     Takes an input DataFrame and returns the computed fraction as a float.
#     """
#     first_login_df = df.groupBy("player_id").agg(
#         F.min("event_date").alias("first_date")
#     )

#     retention_df = first_login_df.join(
#         df,
#         (first_login_df.player_id == df.player_id) &
#         (df.event_date == F.date_add(first_login_df.first_date, 1)),
#         how="left"
#     )

#     result = retention_df.select(
#         F.coalesce(
#             F.round(
#                 F.count_distinct(df.player_id) / F.count_distinct(first_login_df.player_id),
#                 2
#             ),
#             F.lit(0.0)
#         ).alias("fraction")
#     ).collect()[0]["fraction"]

#     return float(result)


def solve_retention(input_df):
  retention_window = Window.partitionBy("player_id").orderBy("event_date")

  lead_df = input_df.withColumn("next_event_date", F.lead("event_date",1).over(retention_window))
  lead_df = lead_df.withColumn("next_day_login",
    F.when(F.datediff(F.col("next_event_date"), F.col("event_date")) == 1, F.col("next_event_date"))
    .otherwise(F.lit(None)) # Set to None if not exactly 1 day later
  )

  count_df = lead_df.groupBy("player_id").agg(
      F.countDistinct("player_id").alias("first_login"),
      F.countDistinct("next_day_login").alias("next_day_login")
  ).replace(0,None, subset = 'next_day_login')

  # Dynamically count non-nulls for all columns
  non_null_counts = count_df.select([F.count(F.col(c)).alias(c) for c in ['first_login', 'next_day_login']])

  result_df = non_null_counts.withColumn("fraction", F.round(F.col("next_day_login") / F.col("first_login"), 2))
  result_df = result_df.select("fraction")

  return result_df.collect()[0]["fraction"]

# -------------------------------------------------------------------------
# Test Runner
# -------------------------------------------------------------------------
print("Running Test Suite...\n" + "="*50)
all_passed = True

for tc in test_cases:
    df_raw = spark.createDataFrame(tc["data"], schema=schema)
    df_test = df_raw.withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))

    actual = solve_retention(df_test)
    passed = (actual == tc["expected"])

    status = "PASSED" if passed else "FAILED"
    print(f"[{status}] {tc['name']}")
    if not passed:
        all_passed = False
        print(f"   -> Expected: {tc['expected']}, Got: {actual}")

print("="*50)
if all_passed:
    print("All test cases passed successfully!")
else:
    print("Some test cases failed. Please check the logs above.")

Running Test Suite...
[PASSED] Test Case 1: Standard / Multi-Player Mix (Expected: 0.33)
[PASSED] Test Case 2: 100% Retention (All players return next day) (Expected: 1.0)
[PASSED] Test Case 3: 0% Retention (No player returns on Day 2) (Expected: 0.0)
[PASSED] Test Case 4: Leap Year & Month Rollover (Expected: 0.5)
[PASSED] Test Case 5: New Year Rollover (Expected: 1.0)
[FAILED] Test Case 6: Duplicate Logins on Both Days & Later Consecutive Days (Expected: 0.5)
   -> Expected: 0.5, Got: 1.0
Some test cases failed. Please check the logs above.
